# Game Intelligence AI — Environment Setup (No Google Drive)

This notebook prepares Google Colab without mounting or depending on Google Drive.

**Storage policy**
- GitHub: source code, notebooks, configuration, documentation and small test fixtures.
- Hugging Face Hub: model repositories, LoRA adapters and datasets when their terms and free quotas permit.
- `/content`: temporary Colab workspace/cache. It is deleted when the runtime resets or expires.
- Google Drive: not used.

GitHub is not unlimited storage for model weights or large datasets. Do not commit secrets, checkpoints or large artifacts to an ordinary Git repository. Upload important outputs to a suitable persistent destination and verify the upload before ending a Colab session.

Run cells from top to bottom. GPU availability and session duration are not guaranteed on free Colab.

## 0. Runtime settings

In Colab choose **Runtime → Change runtime type → T4 GPU** if available, then Save. Do not replace the preinstalled PyTorch/CUDA packages in this setup notebook.

In [ ]:
import os, sys, platform, subprocess, shutil, json
from pathlib import Path
from datetime import datetime, timezone

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Working directory:", os.getcwd())
print("Colab detected:", "COLAB_RELEASE_TAG" in os.environ)

## 1. Configure GitHub repository

Default repository: `t-rex01234/GameIntelligenceAI`. For a private repository, add a Colab Secret named `GH_TOKEN`; never paste or print tokens.

In [ ]:
REPO_URL = "https://github.com/t-rex01234/GameIntelligenceAI.git"
PROJECT_ROOT = Path("/content/GameIntelligenceAI")

if not (PROJECT_ROOT / ".git").exists():
    PROJECT_ROOT.parent.mkdir(parents=True, exist_ok=True)
    clone_url = REPO_URL
    try:
        from google.colab import userdata
        gh_token = userdata.get("GH_TOKEN")
    except Exception:
        gh_token = None
    if gh_token:
        from urllib.parse import urlparse
        u = urlparse(REPO_URL)
        clone_url = f"https://x-access-token:{gh_token}@{u.netloc}{u.path}"
    p = subprocess.run(["git", "clone", clone_url, str(PROJECT_ROOT)],
                       text=True, capture_output=True)
    if p.returncode:
        print(p.stderr)
        raise RuntimeError("Git clone failed. Check repository URL, visibility, branch, and GH_TOKEN if private.")
else:
    print("Repository already cloned.")

os.chdir(PROJECT_ROOT)
print("Project:", PROJECT_ROOT)
print("Files:", sorted(x.name for x in PROJECT_ROOT.iterdir())[:40])

## 2. Create project folders

In [ ]:
FOLDERS = [
 "notebooks", "datasets/raw", "datasets/processed", "datasets/train",
 "datasets/validation", "datasets/test", "models/base", "models/adapters",
 "models/releases", "knowledge_base/godot", "knowledge_base/unity",
 "knowledge_base/unreal", "knowledge_base/game_programming",
 "agents/architect", "agents/programmer", "agents/designer",
 "agents/tester", "agents/orchestrator", "benchmarks", "evaluation",
 "app", "configs", "documentation", "logs", "backups", "tmp"
]
for folder in FOLDERS:
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)
print("Verified", len(FOLDERS), "folders.")

## 3. Check GPU and temporary Colab disk

In [ ]:
def cmd_output(args):
    p = subprocess.run(args, capture_output=True, text=True)
    return p.stdout.strip() if p.returncode == 0 else (p.stderr.strip() or "Unavailable")

print(cmd_output(["bash", "-lc", "nvidia-smi"]))
total, used, free = shutil.disk_usage("/content")
print("Disk total GB:", round(total/1024**3, 2))
print("Disk used GB:", round(used/1024**3, 2))
print("Disk free GB:", round(free/1024**3, 2))

## 4. Install libraries

This deliberately does not install or replace `torch` or CUDA. If dependency conflicts appear, stop and share the full error rather than repeatedly reinstalling packages.

In [ ]:
%pip install -q --upgrade transformers datasets accelerate peft trl bitsandbytes huggingface_hub safetensors sentence-transformers faiss-cpu gradio pytest jsonlines

## 5. Verify package versions and imports

In [ ]:
import importlib, importlib.metadata as metadata

packages = ["torch","transformers","datasets","accelerate","peft","trl",
 "bitsandbytes","huggingface-hub","safetensors","sentence-transformers",
 "faiss-cpu","gradio","pytest","jsonlines"]
package_status = {}
for name in packages:
    try: package_status[name] = metadata.version(name)
    except metadata.PackageNotFoundError: package_status[name] = "NOT INSTALLED"
for name, version in package_status.items(): print(f"{name}: {version}")

modules = ["torch","transformers","datasets","accelerate","peft","trl",
 "bitsandbytes","huggingface_hub","safetensors","sentence_transformers",
 "faiss","gradio","pytest","jsonlines"]
import_status = {}
for name in modules:
    try:
        importlib.import_module(name)
        import_status[name] = "OK"
    except Exception as e:
        import_status[name] = f"ERROR: {type(e).__name__}: {e}"
for name, status in import_status.items(): print(f"{name}: {status}")

## 6. Verify PyTorch and GPU

In [ ]:
import torch
cuda_available = torch.cuda.is_available()
gpu_name, gpu_memory_gb = None, None
print("PyTorch:", torch.__version__)
print("CUDA available:", cuda_available)
print("CUDA build:", torch.version.cuda)
if cuda_available:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory_gb = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
    print("GPU:", gpu_name)
    print("GPU memory GB:", gpu_memory_gb)
    test = torch.randn((512,512), device="cuda")
    print("CUDA smoke test:", tuple((test @ test).shape), "PASSED")
else:
    print("No GPU detected. Fine-tuning should wait until a GPU runtime is available.")

## 7. Optional Hugging Face authentication

Public model metadata usually does not require a token. For gated/private repositories or uploads, add a Colab Secret named `HF_TOKEN` and grant notebook access. The token is never printed or saved in this notebook.

In [ ]:
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Hugging Face authentication available (token hidden).")
else:
    print("No HF_TOKEN configured; public resources can still be accessed.")

## 8. Inspect baseline model metadata (no model download)

`Qwen/Qwen2.5-Coder-7B-Instruct` is only an initial benchmark candidate, not a claim that it is the world's best model. Later, benchmark multiple current candidates against the GPU actually provided.

In [ ]:
from huggingface_hub import model_info
BASE_MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"
try:
    info = model_info(BASE_MODEL_ID, token=HF_TOKEN)
    print("Model:", info.id)
    print("Pipeline:", info.pipeline_tag)
    print("Library:", info.library_name)
    print("Gated:", info.gated)
    print("Last modified:", info.last_modified)
except Exception as e:
    print("Metadata lookup failed:", type(e).__name__, str(e)[:400])

## 9. Test local storage (no Drive)

In [ ]:
test_file = PROJECT_ROOT / "tmp" / "write_test.txt"
test_file.write_text("Game Intelligence AI local storage test.\n", encoding="utf-8")
assert "storage test" in test_file.read_text(encoding="utf-8")
print("Local write/read test: PASSED")

## 10. Check Git configuration

No changes are pushed automatically. Configure your own Git identity when you are ready to commit.

In [ ]:
def git_out(args):
    p = subprocess.run(["git"] + args, cwd=PROJECT_ROOT, capture_output=True, text=True)
    return p.stdout.strip() if p.returncode == 0 else p.stderr.strip()
print("Branch:", git_out(["branch", "--show-current"]))
print("Remotes:\n", git_out(["remote", "-v"]))
print("Git status:\n", git_out(["status", "--short"]))
print("user.name:", git_out(["config", "--get", "user.name"]) or "not configured")
print("user.email:", git_out(["config", "--get", "user.email"]) or "not configured")

## 11. Create a protective .gitignore if missing

In [ ]:
gitignore = PROJECT_ROOT / ".gitignore"
template = '''
.env
.env.*
*.pem
*.key
*token*
*secret*
__pycache__/
*.py[cod]
.venv/
venv/
logs/
tmp/
.cache/
.ipynb_checkpoints/
datasets/raw/
datasets/processed/
datasets/train/
datasets/validation/
datasets/test/
models/base/
models/adapters/
models/releases/
*.safetensors
*.bin
*.pt
*.pth
*.ckpt
*.gguf
'''
if not gitignore.exists():
    gitignore.write_text(template.strip() + "\n", encoding="utf-8")
    print("Created .gitignore")
else:
    print("Existing .gitignore left unchanged.")

## 12. Save environment report locally

In [ ]:
report = {
 "project": "Game Intelligence AI",
 "created_utc": datetime.now(timezone.utc).isoformat(),
 "python": sys.version,
 "platform": platform.platform(),
 "project_root": str(PROJECT_ROOT),
 "gpu_available": bool(cuda_available),
 "gpu_name": gpu_name,
 "gpu_memory_gb": gpu_memory_gb,
 "torch_version": torch.__version__,
 "cuda_build": torch.version.cuda,
 "packages": package_status,
 "imports": import_status,
 "colab_disk_gb": {"total": round(total/1024**3,2),
                   "used": round(used/1024**3,2),
                   "free": round(free/1024**3,2)}
}
report_json = PROJECT_ROOT / "logs" / "environment_report.json"
report_txt = PROJECT_ROOT / "logs" / "environment_report.txt"
report_json.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
report_txt.write_text("\n".join(f"{k}: {v}" for k,v in report.items()), encoding="utf-8")
print("Saved:", report_json)
print("Saved:", report_txt)

## 13. Download the report (optional)

In [ ]:
try:
    from google.colab import files
    files.download(str(report_json))
except Exception as e:
    print("Colab download helper unavailable:", e)

## 14. Final readiness check

Do not start fine-tuning from this notebook. The next step is a model benchmark and memory check before downloading model weights.

Before a Colab session ends:
1. Commit reviewed source/notebook changes to GitHub.
2. Upload important datasets/adapters/checkpoints to a suitable persistent destination, such as Hugging Face Hub when appropriate.
3. Verify the upload by listing or downloading the files.
4. Keep credentials in Colab Secrets, never in notebook cells or Git.

Anything under `/content` is temporary. GitHub is for code and small project files, not a free unlimited model/data drive.

In [ ]:
checks = {
 "repository_present": (PROJECT_ROOT / ".git").exists(),
 "folders_present": all((PROJECT_ROOT / f).exists() for f in FOLDERS),
 "local_storage_test": test_file.exists(),
 "report_created": report_json.exists(),
 "torch_import": import_status.get("torch") == "OK",
 "transformers_import": import_status.get("transformers") == "OK",
 "gpu_available": bool(cuda_available)
}
for k,v in checks.items(): print(("PASS" if v else "CHECK"), "-", k)
print("\nResult:", "READY FOR MODEL BENCHMARKING" if all(checks.values()) else "PARTIAL SETUP — inspect CHECK items.")